# QQQI ↔ QQQ ↔ TQQQ 日线状态机回测

**目的**：检验 QQQI 在弱市/震荡市中的防守价值、QQQ 在修复阶段的恢复速度，以及基于 QQQ 的 MA200 + MA20/布林信号在 QQQI、QQQ、TQQQ 之间切换后，是否改善 CAGR、最大回撤、Sharpe、Sortino 与 Calmar。

**研究边界**

- 所有信号只使用 QQQ 在 `t` 日收盘时已知的数据。
- `t` 日收盘决定目标状态，`t+1` 日开盘执行。
- 收益按复权开盘价的 open-to-open 计算；换仓按“卖出 + 买入”两条腿扣费。
- QQQI 官方成立日期为 2024-01-29，因此三标的真实共同样本无法覆盖 2020 与 2022。Notebook 会明确标记这些阶段“共同历史不足”，不会伪造 QQQI 历史或默认使用代理。
- 默认参数是先验主检验；参数网格只做稳健性诊断，不用于回看后挑选最优组合。

> Research only. `trade_ready = false`.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.research.etf_rotation_experiment import (
    RotationConfig,
    chronological_split_metrics,
    conditional_asset_metrics,
    fetch_adjusted_daily_bars,
    phase_metrics,
    recovery_event_study,
    run_default_comparison,
    run_sensitivity_grid,
    stability_summary,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

## 1. 冻结实验合同

先读取版本化 YAML。后续默认回测不允许根据结果临时修改参数；修改参数意味着新实验合同。

In [ ]:
CONTRACT_PATH = ROOT / "configs/research_paradigms/qqqi_qqq_tqqq_rotation_v1.yaml"
contract = yaml.safe_load(CONTRACT_PATH.read_text(encoding="utf-8"))
config = RotationConfig(**contract["strategy"])

print(json.dumps({
    "experiment_id": contract["experiment_id"],
    "research_only": contract["research_only"],
    "trade_ready": contract["trade_ready"],
    "execution": contract["boundaries"]["execution_time"],
    "return_measurement": contract["boundaries"]["return_measurement"],
    "strategy": contract["strategy"],
}, ensure_ascii=False, indent=2))

## 2. 下载并审计数据

复用 AlphaEngine 的 `YFinanceAdapter`。该适配器请求自动复权 OHLC，因此开盘价与收盘价处于一致的公司行动调整口径。

可通过 `END_DATE` 冻结研究截止日。留空则取数据源可用的最新完整日线。

In [ ]:
END_DATE = contract["data"].get("end_date")  # 例如 "2026-07-31"；None 表示最新
bars, coverage = fetch_adjusted_daily_bars(
    symbols=contract["boundaries"]["tradable_symbols"],
    start=contract["data"]["start_date"],
    end=END_DATE,
)
coverage

### 数据边界检查

主策略必须采用三只 ETF 的共同可交易区间；但 QQQ 的 MA200/MA20 指标会利用共同区间之前的 QQQ 历史进行预热。

In [ ]:
metrics, results, prepared = run_default_comparison(bars, config)

sample_audit = pd.Series({
    "common_post_warmup_start": prepared.index.min().date().isoformat(),
    "common_post_warmup_end": prepared.index.max().date().isoformat(),
    "common_sessions_including_last_open": len(prepared),
    "economic_return_sessions": results["rotation_B"].metrics["observations"],
    "qqqi_first_observation": coverage.set_index("symbol").loc["QQQI", "first_date"],
})
sample_audit

## 3. 默认参数主检验

比较五组：

1. Buy & Hold QQQI
2. Buy & Hold QQQ
3. Buy & Hold TQQQ
4. Rotation A：QQQI ↔ QQQ
5. Rotation B：QQQI ↔ QQQ ↔ TQQQ

所有策略使用相同起止日期、相同复权口径与一致的初始买入成本。

In [ ]:
metric_columns = [
    "total_return", "cagr", "annual_volatility", "sharpe", "sortino",
    "max_drawdown", "calmar", "switch_count", "average_holding_days",
    "transaction_cost_paid", "pct_time_qqqi", "pct_time_qqq", "pct_time_tqqq",
]
metrics[metric_columns].sort_values("calmar", ascending=False)

### 净值曲线

In [ ]:
equity = pd.concat(
    {name: result.daily["equity"] for name, result in results.items()}, axis=1
).dropna(how="all")
ax = equity.plot(figsize=(13, 6), title="Common-window equity curves (adjusted open-to-open)")
ax.set_ylabel("Growth of 1.0")
ax.set_xlabel("")
ax.grid(True, alpha=0.25)
plt.show()

### 回撤曲线

In [ ]:
drawdowns = pd.concat(
    {name: result.daily["drawdown"] for name, result in results.items()}, axis=1
).dropna(how="all")
ax = drawdowns.plot(figsize=(13, 6), title="Drawdown comparison")
ax.set_ylabel("Drawdown")
ax.set_xlabel("")
ax.grid(True, alpha=0.25)
plt.show()

## 4. 状态切换审计

`decision_state` 是当日收盘后产生的目标状态；`position_state` 必须严格等于前一交易日的 `decision_state`。这张表和时间轴用于人工核验未来函数与切换原因。

In [ ]:
rotation_b = results["rotation_B"]
assert (
    rotation_b.daily["position_state"]
    == rotation_b.daily["decision_state"].shift(1).fillna(0).astype(int)
).all()

rotation_b.trades.tail(30)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 3.5))
ax.step(rotation_b.daily.index, rotation_b.daily["position_state"], where="post")
ax.set_yticks([0, 1, 2], ["QQQI", "QQQ", "TQQQ"])
ax.set_title("Rotation B executed state timeline")
ax.set_xlabel("")
ax.grid(True, axis="x", alpha=0.25)
plt.show()

## 5. 核心前提一：QQQI 是否在弱市/震荡市更抗跌？

市场状态只由 QQQ 定义：

- `weak_below_ma200`：QQQ 收盘低于 MA200；
- `sideways_above_ma200`：QQQ 在 MA200 上方且 63 日收益绝对值小于 5%；
- `uptrend`：QQQ 在 MA200 上方、63 日收益至少 5%，且 MA20 连续上行；
- 其余为 `transition`。

重点比较 QQQI 与 QQQ 的累计收益、年化波动、Sharpe 和最大回撤。

In [ ]:
conditional = conditional_asset_metrics(prepared)
conditional

In [ ]:
weak = conditional.loc["weak_below_ma200"]
sideways = conditional.loc["sideways_above_ma200"]

def defensive_readout(table, regime_name):
    if table["sessions"].min() < 20:
        return f"{regime_name}: 样本不足，不能形成稳定判断。"
    vol_better = table.loc["QQQI", "annualized_volatility"] < table.loc["QQQ", "annualized_volatility"]
    dd_better = abs(table.loc["QQQI", "max_drawdown"]) < abs(table.loc["QQQ", "max_drawdown"])
    ret_better = table.loc["QQQI", "cumulative_return"] > table.loc["QQQ", "cumulative_return"]
    return (
        f"{regime_name}: QQQI 波动更低={vol_better}，最大回撤更小={dd_better}，"
        f"累计收益更高={ret_better}。"
    )

print(defensive_readout(weak, "弱市"))
print(defensive_readout(sideways, "震荡市"))

## 6. 核心前提二：QQQ 是否在修复阶段恢复更快？

事件定义：QQQ 从 MA200 下方重新收于 MA200 上方。比较事件后 20 个交易日 QQQ 与 QQQI 的 open-to-open 累计收益。重叠事件仍会列出，便于人工检查；由于 QQQI 样本很短，事件数可能很少。

In [ ]:
recovery = recovery_event_study(
    prepared,
    horizon_sessions=contract["validation"]["recovery_event_horizon_sessions"],
)
recovery

In [ ]:
if recovery.empty:
    print("共同样本内没有足够完整的 MA200 修复事件，无法回答恢复速度问题。")
else:
    summary = recovery[["QQQI_return", "QQQ_return", "QQQ_minus_QQQI"]].agg(
        ["count", "mean", "median", "min", "max"]
    )
    display(summary)
    print(
        "QQQ 在修复事件中跑赢 QQQI 的比例:",
        f"{(recovery['QQQ_minus_QQQI'] > 0).mean():.1%}",
    )

## 7. 指定历史阶段

2020、2022 等阶段会被保留在表中，但如果早于 QQQI 共同历史，必须显示 `insufficient_common_history`。这正是研究结论的一部分，而不是数据错误。

In [ ]:
periods = {
    name: (dates[0], dates[1])
    for name, dates in contract["named_periods"].items()
}
phase_table = phase_metrics(
    results,
    periods,
    minimum_sessions=contract["validation"]["minimum_phase_sessions"],
)
phase_table

## 8. 时间切分验证

QQQI 不具备 2010–2020 / 2021–2026 的共同样本。这里不做伪样本内外优化，而是把同一套冻结参数原样应用于共同历史的前 60% 与后 40%，检查后段是否明显恶化。

In [ ]:
split = chronological_split_metrics(
    results["rotation_B"],
    train_fraction=contract["validation"]["chronological_train_fraction"],
)
split

In [ ]:
early = split.loc["early_common_sample"]
late = split.loc["late_common_sample"]
checks = {
    "late_cagr_at_least_70pct_of_early": (
        late["cagr"] >= 0.70 * early["cagr"] if early["cagr"] > 0 else np.nan
    ),
    "late_mdd_not_over_1_3x_early": (
        abs(late["max_drawdown"]) <= 1.30 * abs(early["max_drawdown"])
        if early["max_drawdown"] < 0 else np.nan
    ),
}
checks

## 9. 参数敏感性

完整合同为 3 × 4 × 3 × 3 × 3 = 324 组。默认运行全部组合；机器资源有限时可先把 `RUN_FULL_GRID=False`，但最终结论应使用完整网格。

**注意**：下面不把最高 Calmar 组合称为“最优参数”。重点看分布、邻域稳定性以及默认参数是否处于正常区域。

In [ ]:
RUN_FULL_GRID = True

if RUN_FULL_GRID:
    grid = run_sensitivity_grid(bars, config, contract["sensitivity"], version="B")
else:
    compact_grid = {
        "ma_long": [180, 200, 220],
        "buffer": [0.005, 0.01, 0.02],
        "n_rise": [3, 5],
        "drawdown_threshold": [0.08, 0.10, 0.15],
        "n_exit_short": [1, 2],
    }
    grid = run_sensitivity_grid(bars, config, compact_grid, version="B")

print("parameter combinations:", len(grid))
grid[[
    "ma_long", "buffer", "n_rise", "drawdown_threshold", "n_exit_short",
    "cagr", "max_drawdown", "calmar", "sharpe", "switch_count",
]].describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90])

In [ ]:
stability = stability_summary(
    grid,
    baseline_metrics=results["rotation_B"].metrics,
)
stability

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(grid["max_drawdown"].abs(), grid["cagr"], alpha=0.55)
ax.scatter(
    [abs(results["rotation_B"].metrics["max_drawdown"])],
    [results["rotation_B"].metrics["cagr"]],
    marker="X",
    s=160,
    label="Frozen default",
)
ax.set_xlabel("Absolute maximum drawdown")
ax.set_ylabel("CAGR")
ax.set_title("Sensitivity cloud: return vs drawdown")
ax.legend()
ax.grid(True, alpha=0.25)
plt.show()

## 10. 自动结论框架

自动输出只回答数据能支持的内容，并把短样本、事件不足、参数不稳健等限制放在结论中。最终投资判断仍需人工审阅交易列表、数据覆盖和异常日期。

In [ ]:
def pct(value):
    return "N/A" if pd.isna(value) else f"{value:.2%}"

def num(value):
    return "N/A" if pd.isna(value) else f"{value:.2f}"

qqq = results["buy_hold_QQQ"].metrics
qqqi = results["buy_hold_QQQI"].metrics
tqqq = results["buy_hold_TQQQ"].metrics
rot = results["rotation_B"].metrics

print("共同样本:", rot["start_date"], "至", rot["end_date"])
print(
    "Rotation B vs QQQ — CAGR:", pct(rot["cagr"]), "vs", pct(qqq["cagr"]),
    "| MDD:", pct(rot["max_drawdown"]), "vs", pct(qqq["max_drawdown"]),
    "| Calmar:", num(rot["calmar"]), "vs", num(qqq["calmar"]),
)
print(
    "Rotation B vs QQQI — CAGR:", pct(rot["cagr"]), "vs", pct(qqqi["cagr"]),
    "| MDD:", pct(rot["max_drawdown"]), "vs", pct(qqqi["max_drawdown"]),
)
print(
    "Rotation B vs TQQQ — CAGR:", pct(rot["cagr"]), "vs", pct(tqqq["cagr"]),
    "| MDD:", pct(rot["max_drawdown"]), "vs", pct(tqqq["max_drawdown"]),
)
print("默认策略切换次数:", rot["switch_count"], "平均持有天数:", num(rot["average_holding_days"]))
print("参数稳健性启发式通过:", stability["heuristic_robust"])
print()
print("必须保留的限制：")
print("1. QQQI 成立于 2024-01-29，无法直接验证 2020 疫情与 2022 加息熊市。")
print("2. 共同样本较短，独立牛熊周期数量有限，Sharpe/Calmar 误差很大。")
print("3. TQQQ 的阈值含义是‘回撤后的趋势修复加杠杆’，不是创新高时持续加杠杆。")
print("4. 网格仅检验参数邻域，不允许从中回看挑选赢家后直接投入交易。")

## 11. 可复现实验输出

完整证据包建议通过 CLI 生成，它会写出覆盖报告、每日净值、交易记录、分阶段结果、敏感性网格和 SHA-256 manifest：

```bash
uv run python scripts/run_qqqi_qqq_tqqq_rotation.py
```

需要冻结截止日时：

```bash
uv run python scripts/run_qqqi_qqq_tqqq_rotation.py --end-date 2026-07-31
```
